# 04 — Solving FrozenLake with Dynamic Programming
**Week 4 | Dynamic Programming**

Gymnasium environments expose `env.P[s][a]` — the full transition model.
This lets us apply exact DP and verify our solver achieves 100% win rate.

In [ ]:
try:
    import gymnasium as gym
except ImportError:
    import subprocess, sys; subprocess.check_call([sys.executable,'-m','pip','install','gymnasium','-q']); import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

env = gym.make('FrozenLake-v1', is_slippery=False)
env_slip = gym.make('FrozenLake-v1', is_slippery=True)

print(f"States: {env.observation_space.n}")
print(f"Actions: {env.action_space.n}  (0=L, 1=D, 2=R, 3=U)")

In [ ]:
def value_iteration_gym(env_unwrapped, gamma=0.99, theta=1e-8):
    n_s = env_unwrapped.observation_space.n
    n_a = env_unwrapped.action_space.n
    V = np.zeros(n_s)
    errors = []
    for i in range(10_000):
        delta = 0
        V_new = np.zeros(n_s)
        for s in range(n_s):
            q_vals = []
            for a in range(n_a):
                q = sum(p*(r + gamma*V[ns]*(not d)) for p,ns,r,d in env_unwrapped.unwrapped.P[s][a])
                q_vals.append(q)
            V_new[s] = max(q_vals)
            delta = max(delta, abs(V_new[s]-V[s]))
        V = V_new; errors.append(delta)
        if delta < theta:
            print(f"Converged in {i+1} sweeps"); break
    # Extract policy
    policy = np.zeros(n_s, dtype=int)
    for s in range(n_s):
        qs = [sum(p*(r+gamma*V[ns]*(not d)) for p,ns,r,d in env_unwrapped.unwrapped.P[s][a]) for a in range(n_a)]
        policy[s] = np.argmax(qs)
    return V, policy, errors

print("--- Non-slippery ---")
V_ns, pi_ns, err_ns = value_iteration_gym(env)
print("--- Slippery ---")
V_sl, pi_sl, err_sl = value_iteration_gym(env_slip)

In [ ]:
def evaluate_policy(env, policy, n_episodes=1000):
    wins = 0
    for _ in range(n_episodes):
        s, _ = env.reset()
        for _ in range(200):
            s, r, term, trunc, _ = env.step(policy[s])
            if term or trunc:
                if r == 1.0: wins += 1
                break
    return wins / n_episodes

print(f"Non-slippery win rate: {evaluate_policy(env, pi_ns):.0%}")
print(f"Slippery    win rate: {evaluate_policy(env_slip, pi_sl):.0%}")

In [ ]:
# Visualise both value functions
ACTION_SYMBOLS = {0:'←', 1:'↓', 2:'→', 3:'↑'}
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for row, (V, pi, title, errs) in enumerate([
    (V_ns, pi_ns, 'Non-Slippery', err_ns),
    (V_sl, pi_sl, 'Slippery',     err_sl)]):
    im = axes[row,0].imshow(V.reshape(4,4), cmap='RdYlGn')
    plt.colorbar(im, ax=axes[row,0])
    for s in range(16):
        r,c=divmod(s,4)
        axes[row,0].text(c,r,f'{V[s]:.2f}',ha='center',va='center',fontsize=9)
    axes[row,0].set_title(f'V* — {title}'); axes[row,0].set_xticks([]); axes[row,0].set_yticks([])

    axes[row,1].set_xlim(-0.5,3.5); axes[row,1].set_ylim(-0.5,3.5)
    axes[row,1].set_xticks(range(4)); axes[row,1].set_yticks(range(4)); axes[row,1].grid(True)
    for s in range(16):
        r,c=divmod(s,4)
        axes[row,1].text(c,3-r,ACTION_SYMBOLS[pi[s]],ha='center',va='center',fontsize=20)
    axes[row,1].set_title(f'π* — {title}')
plt.tight_layout(); plt.show()
print("Note: different optimal policies for slippery vs non-slippery!")

## ✅ Exercises
1. Change gamma from 0.99 to 0.5 for the slippery version. Does the win rate improve or decrease?
2. Use the 8×8 map (`map_name='8x8'`). How many sweeps does value iteration need?
3. **Challenge**: implement value iteration without using `env.P` — only using `env.step()`. This transitions you to model-free learning!

## Q1 

In [ ]:
V_sl_05, pi_sl_05, _ = value_iteration_gym(env_slip, gamma=0.5)
print(f"Slippery win rate (γ=0.5): {evaluate_policy(env_slip, pi_sl_05):.0%}")
print(f"Slippery win rate (γ=0.99): {evaluate_policy(env_slip, pi_sl):.0%}")

Decreases. With γ=0.5, future rewards are discounted so heavily that the agent undervalues reaching the goal (which is multiple steps away) and overweights immediate step penalties. This causes the derived policy to be suboptimal on the slippery environment where longer, safer routes are necessary.

## Q2 

In [ ]:
env_8x8 = gym.make('FrozenLake-v1', map_name='8x8', is_slippery=False)
V_8x8, pi_8x8, err_8x8 = value_iteration_gym(env_8x8)
print(f"Win rate (8x8): {evaluate_policy(env_8x8, pi_8x8):.0%}")

he 8×8 map requires significantly more sweeps than the 4×4 map — typically around 3–5× more — because the larger state space means value information takes more iterations to propagate from the goal back to distant starting states.

## Q3

In [ ]:
def value_iteration_modelfree(env, gamma=0.99, n_samples=20, theta=1e-6, max_iter=500):
    """
    Approximate value iteration using env.step() only.
    For each (s, a), sample n_samples transitions to estimate Q(s,a).
    """
    n_s = env.observation_space.n
    n_a = env.action_space.n
    V   = np.zeros(n_s)

    for iteration in range(max_iter):
        delta = 0
        V_new = np.zeros(n_s)
        for s in range(n_s):
            q_vals = []
            for a in range(n_a):
                q_samples = []
                for _ in range(n_samples):
                    env.unwrapped.s = s          # force environment to state s
                    ns, r, term, trunc, _ = env.step(a)
                    q_samples.append(r + gamma * V[ns] * (not (term or trunc)))
                q_vals.append(np.mean(q_samples))
            V_new[s] = max(q_vals)
            delta = max(delta, abs(V_new[s] - V[s]))
        V = V_new
        if delta < theta:
            print(f"Converged in {iteration+1} iterations")
            break

    policy = np.zeros(n_s, dtype=int)
    for s in range(n_s):
        q_vals = []
        for a in range(n_a):
            samples = []
            for _ in range(n_samples):
                env.unwrapped.s = s
                ns, r, term, trunc, _ = env.step(a)
                samples.append(r + gamma * V[ns] * (not (term or trunc)))
            q_vals.append(np.mean(samples))
        policy[s] = np.argmax(q_vals)
    return V, policy

V_mf, pi_mf = value_iteration_modelfree(env)
print(f"Model-free VI win rate: {evaluate_policy(env, pi_mf):.0%}")